# ELSST Track 1: Hierarchy-Aware Retrieval

This notebook implements hierarchy-aware retrieval using ELSST's thesaurus structure motivated by Wang et al.'s Riemannian
ranking work showing that flat embedding similarity misses the hierarchical structure
that gives concepts like these their meaning.

## 1. Environment Setup
> Mounting Drive and loading everything needed: dataset, concept pool, Qwen3 embeddings
already computed and saved from the baselines notebook, and evaluation functions.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
import numpy as np

project_path = '/content/drive/MyDrive/ELSST_Project'
RESULTS_DIR = os.path.join(project_path, 'results')
CHECKPOINT_DIR = os.path.join(project_path, 'checkpoints')

from datasets import load_from_disk
from huggingface_hub import hf_hub_download

dataset = load_from_disk(os.path.join(project_path, 'dataset'))

pool_path = hf_hub_download(repo_id="JohnWang10086/elsst-track1", filename="concept_pool.jsonl", repo_type="dataset")
concept_pool_lookup = {}
with open(pool_path, "r") as f:
    for line in f:
        c = json.loads(line)
        concept_pool_lookup[c["concept_id"]] = c

concept_ids = list(concept_pool_lookup.keys())
val_ids = [ex['id'] for ex in dataset['validation']]
gold = {ex['id']: ex['retrieval_labels']['positive_ids'] for ex in dataset['validation']}

with open(os.path.join(RESULTS_DIR, 'qwen_predictions.json')) as f:
    qwen_predictions = json.load(f)

print(f"Loaded {len(concept_pool_lookup)} concepts, {len(val_ids)} validation docs")

Mounted at /content/drive


concept_pool.jsonl:   0%|          | 0.00/800k [00:00<?, ?B/s]

Loaded 3433 concepts, 756 validation docs


In [3]:
def reciprocal_rank(ranked_ids, positive_ids):
    positive_set = set(positive_ids)
    for rank, cid in enumerate(ranked_ids, start=1):
        if cid in positive_set:
            return 1.0 / rank
    return 0.0

def recall_at_k(ranked_ids, positive_ids, k):
    if not positive_ids:
        return 0.0
    top_k = set(ranked_ids[:k])
    return len(top_k & set(positive_ids)) / len(positive_ids)

def ndcg_at_k(ranked_ids, positive_ids, k=10):
    positive_set = set(positive_ids)
    dcg = sum(1.0 / np.log2(i + 1) for i, cid in enumerate(ranked_ids[:k], start=1) if cid in positive_set)
    ideal_hits = min(len(positive_ids), k)
    idcg = sum(1.0 / np.log2(i + 1) for i in range(1, ideal_hits + 1))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_retrieval(predictions, gold):
    mrr, r5, r10, ndcg = [], [], [], []
    for doc_id, positive_ids in gold.items():
        ranked = predictions[doc_id]
        mrr.append(reciprocal_rank(ranked, positive_ids))
        r5.append(recall_at_k(ranked, positive_ids, 5))
        r10.append(recall_at_k(ranked, positive_ids, 10))
        ndcg.append(ndcg_at_k(ranked, positive_ids, 10))
    return {"MRR": float(np.mean(mrr)), "Recall@5": float(np.mean(r5)), "Recall@10": float(np.mean(r10)), "NDCG@10": float(np.mean(ndcg))}

## 2. Fetch and Parse ELSST Hierarchy
> Downloading the official ELSST thesaurus as SKOS JSON-LD and extracting broader/
narrower/related relationships. Matching to our concept pool is done by term text, not
ID, since the benchmark's concept_pool.jsonl uses its own internal UUIDs rather than
ELSST's canonical concept URIs.

In [4]:
import requests

skos_path = os.path.join(CHECKPOINT_DIR, 'elsst_skos.json')

if os.path.exists(skos_path):
    with open(skos_path) as f:
        skos_data = json.load(f)
    print("Loaded cached SKOS data")
else:
    url = "https://thesauri.cessda.eu/rest/v1/elsst-6/data?format=application%2Fld%2Bjson"
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    skos_data = response.json()
    with open(skos_path, 'w') as f:
        json.dump(skos_data, f)
    print("Downloaded and cached SKOS data")

graph = skos_data[0]["@graph"]
print(f"Total graph entries: {len(graph)}")
print("\nSample entry:")
print(json.dumps(graph[10], indent=2)[:800])

Loaded cached SKOS data
Total graph entries: 3478

Sample entry:
{
  "@id": "https://elsst.cessda.eu/id/6/00298b34-d139-4dfb-a756-2c7eb15d5bba",
  "@type": [
    "http://www.w3.org/2004/02/skos/core#Concept"
  ],
  "http://purl.org/dc/terms/identifier": [
    {
      "@language": "cs",
      "@value": "urn:ddi:int.cessda.elsst:00298b34-d139-4dfb-a756-2c7eb15d5bba:6"
    },
    {
      "@language": "de",
      "@value": "urn:ddi:int.cessda.elsst:00298b34-d139-4dfb-a756-2c7eb15d5bba:6"
    },
    {
      "@language": "el",
      "@value": "urn:ddi:int.cessda.elsst:00298b34-d139-4dfb-a756-2c7eb15d5bba:6"
    },
    {
      "@language": "en",
      "@value": "urn:ddi:int.cessda.elsst:00298b34-d139-4dfb-a756-2c7eb15d5bba:6"
    },
    {
      "@language": "es",
      "@value": "urn:ddi:int.cessda.elsst:00298b34-d139-4dfb-a756-2c7eb15d5bba:6"
    },
    {
   


In [5]:
SKOS = "http://www.w3.org/2004/02/skos/core#"

def get_label(node, predicate):
    vals = node.get(predicate, [])
    if isinstance(vals, dict):
        vals = [vals]
    for v in vals:
        if isinstance(v, dict) and v.get('@language') == 'en':
            return v.get('@value')
    return None

def get_refs(node, predicate):
    vals = node.get(predicate, [])
    if isinstance(vals, dict):
        vals = [vals]
    return [v.get('@id') for v in vals if isinstance(v, dict) and '@id' in v]

concept_nodes = {}
for node in graph:
    label = get_label(node, SKOS + 'prefLabel')
    if label:
        concept_nodes[node['@id']] = {
            'term': label.upper().strip(),
            'broader': get_refs(node, SKOS + 'broader'),
            'narrower': get_refs(node, SKOS + 'narrower'),
            'related': get_refs(node, SKOS + 'related'),
        }

print(f"Concepts with English prefLabel: {len(concept_nodes)}")

sample_id = list(concept_nodes.keys())[0]
print(f"\nSample: {concept_nodes[sample_id]}")

Concepts with English prefLabel: 3471

Sample: {'term': 'ELSST THESAURUS', 'broader': [], 'narrower': [], 'related': []}


In [6]:
term_to_node = {node['term']: node_id for node_id, node in concept_nodes.items()}

matched = 0
unmatched_terms = []
for cid, c in concept_pool_lookup.items():
    if c['term'].upper().strip() in term_to_node:
        matched += 1
    else:
        unmatched_terms.append(c['term'])

print(f"Matched: {matched}/{len(concept_pool_lookup)} ({matched/len(concept_pool_lookup)*100:.1f}%)")
print(f"\nSample unmatched terms: {unmatched_terms[:10]}")

Matched: 3433/3433 (100.0%)

Sample unmatched terms: []


## 3. Hierarchy-Aware Re-ranking
> **Approach:** Take Qwen3's top-50 ranked candidates per document, assign each a base
score from its rank position higher rank = higher score, then boost each candidate's
score based on how many of its ELSST neighbors (broader/narrower/related concepts) also
appear in the same top-50 list. The intuition: if several thematically connected concepts
all show up as candidates, that's a stronger signal of relevance than an isolated concept
with no related concepts nearby directly inspired by the Riemannian paper's argument that
flat similarity misses this structural signal.
>
> **Hypothesis:** Should modestly improve over Qwen3 alone by promoting concepts that are part of a coherent semantic cluster, and demoting isolated false positives that happen to score well on embedding similarity but have no structural support from related concepts.

In [7]:
term_to_concept_id = {c['term'].upper().strip(): cid for cid, c in concept_pool_lookup.items()}
concept_neighbors = {}

for cid, c in concept_pool_lookup.items():
    term = c['term'].upper().strip()
    node_id = term_to_node.get(term)
    if node_id is None:
        concept_neighbors[cid] = set()
        continue

    node = concept_nodes[node_id]
    neighbor_node_ids = node['broader'] + node['narrower'] + node['related']
    neighbor_concept_ids = set()
    for nid in neighbor_node_ids:
        if nid in concept_nodes:
            neighbor_term = concept_nodes[nid]['term']
            if neighbor_term in term_to_concept_id:
                neighbor_concept_ids.add(term_to_concept_id[neighbor_term])
    concept_neighbors[cid] = neighbor_concept_ids

avg_neighbors = np.mean([len(v) for v in concept_neighbors.values()])
print(f"Average neighbors per concept: {avg_neighbors:.1f}")
print(f"Concepts with 0 neighbors: {sum(1 for v in concept_neighbors.values() if len(v) == 0)}")

Average neighbors per concept: 3.6
Concepts with 0 neighbors: 6


### 3.1 Re-ranking function
> Score each candidate as: base rank score + a boost proportional to how many of its ELSST neighbors also appear in the same top-50 candidate list. This rewards concepts that sit in
a cluster of related candidates over isolated ones, without needing raw similarity scores we didn't save from the baselines notebook.

In [8]:
def hierarchy_rerank(candidates, neighbor_boost_weight=0.3):
    n = len(candidates)
    candidate_set = set(candidates)

    base_scores = {cid: (n - i) for i, cid in enumerate(candidates)}

    boosted_scores = {}
    for cid in candidates:
        neighbors_in_list = concept_neighbors.get(cid, set()) & candidate_set
        boost = len(neighbors_in_list) * neighbor_boost_weight * (n / 10)
        boosted_scores[cid] = base_scores[cid] + boost

    return sorted(candidates, key=lambda cid: boosted_scores[cid], reverse=True)

### 3.2 Quick test before the full run

In [9]:
test_doc_id = val_ids[0]
test_candidates = qwen_predictions[test_doc_id]
reranked = hierarchy_rerank(test_candidates)

print(f"Gold: {[concept_pool_lookup[cid]['term'] for cid in gold[test_doc_id]]}")
print(f"Original top 5: {[concept_pool_lookup[cid]['term'] for cid in test_candidates[:5]]}")
print(f"Reranked top 5: {[concept_pool_lookup[cid]['term'] for cid in reranked[:5]]}")

Gold: ['REGIONAL ECONOMY', 'REGIONAL FINANCE']
Original top 5: ['REGIONAL FINANCE', 'LOCAL FINANCE', 'REGIONAL ECONOMY', 'DECENTRALIZED GOVERNMENT', 'LOCAL GOVERNMENT POLICY']
Reranked top 5: ['REGIONAL FINANCE', 'LOCAL FINANCE', 'REGIONAL ECONOMY', 'DECENTRALIZED GOVERNMENT', 'PRIVATE SECTOR']


### 3.3 Full validation run
> Applying hierarchy-aware re-ranking to Qwen3's top-50 for every validation document.

In [10]:
def compute_hierarchy_aware():
    predictions = {}
    for doc_id in val_ids:
        candidates = qwen_predictions[doc_id]
        predictions[doc_id] = hierarchy_rerank(candidates)

    metrics = evaluate_retrieval(predictions, gold)

    with open(os.path.join(RESULTS_DIR, 'hierarchy_aware_predictions.json'), 'w') as f:
        json.dump(predictions, f)

    return metrics

def run_or_load(name, compute_fn):
    path = os.path.join(RESULTS_DIR, f"{name}.json")
    if os.path.exists(path):
        print(f"Loaded cached result: {name}")
        with open(path) as f:
            return json.load(f)
    print(f"Running: {name}")
    result = compute_fn()
    with open(path, "w") as f:
        json.dump(result, f, indent=2)
    return result

hierarchy_metrics = run_or_load('hierarchy_aware_retrieval', compute_hierarchy_aware)
print(hierarchy_metrics)

Loaded cached result: hierarchy_aware_retrieval
{'MRR': 0.32073945615537364, 'Recall@5': 0.2760361552028219, 'Recall@10': 0.3613095238095238, 'NDCG@10': 0.25930430277209204}


### 3.4 Statistical significance vs. Qwen3
> Checking whether the raw co-occurrence boost's apparent drop 0.368 -> 0.321 is a real, reliable effect rather than noise, using the same paired Wilcoxon approach as the prompting notebook.

In [11]:
from scipy.stats import wilcoxon

qwen_rr = [reciprocal_rank(qwen_predictions[doc_id], gold[doc_id]) for doc_id in val_ids]

with open(os.path.join(RESULTS_DIR, 'hierarchy_aware_predictions.json')) as f:
    hierarchy_predictions = json.load(f)
hierarchy_rr = [reciprocal_rank(hierarchy_predictions[doc_id], gold[doc_id]) for doc_id in val_ids]

stat, p = wilcoxon(qwen_rr, hierarchy_rr)
changed = sum(1 for doc_id in val_ids if hierarchy_predictions[doc_id] != qwen_predictions[doc_id])

print(f"Qwen3 MRR: {np.mean(qwen_rr):.4f}")
print(f"Hierarchy-aware MRR: {np.mean(hierarchy_rr):.4f}")
print(f"Documents where ranking changed: {changed}/{len(val_ids)}")
print(f"Wilcoxon p-value: {p:.4f}")

Qwen3 MRR: 0.3678
Hierarchy-aware MRR: 0.3207
Documents where ranking changed: 756/756
Wilcoxon p-value: 0.0000


### 3.5 Diagnosing the failure: a traced example
> Tracing the single largest drop from Qwen3 to hierarchy-aware re-ranking to understand why the boost hurts, rather than just accepting the aggregate number this surfaces the exact failure mode driving the drop.

In [12]:
worst_drop_id = None
worst_drop = 0
for doc_id in val_ids:
    drop = reciprocal_rank(qwen_predictions[doc_id], gold[doc_id]) - reciprocal_rank(hierarchy_predictions[doc_id], gold[doc_id])
    if drop > worst_drop:
        worst_drop = drop
        worst_drop_id = doc_id

doc_id = worst_drop_id
qwen_ranked = qwen_predictions[doc_id]
hier_ranked = hierarchy_predictions[doc_id]
gold_ids = gold[doc_id]

print(f"Document: {doc_id}")
print(f"Gold: {[concept_pool_lookup[cid]['term'] for cid in gold_ids]}")

for gid in gold_ids:
    qwen_rank = qwen_ranked.index(gid) + 1 if gid in qwen_ranked else "not in top 50"
    hier_rank = hier_ranked.index(gid) + 1 if gid in hier_ranked else "not in top 50"
    print(f"\n'{concept_pool_lookup[gid]['term']}': Qwen3 rank {qwen_rank} -> Hierarchy-aware rank {hier_rank}")

print(f"\nQwen3 top 5: {[concept_pool_lookup[c]['term'] for c in qwen_ranked[:5]]}")
print(f"Hierarchy-aware top 5: {[concept_pool_lookup[c]['term'] for c in hier_ranked[:5]]}")

print("\nNeighbor-in-list counts for hierarchy-aware top 5:")
candidate_set = set(qwen_ranked)
for c in hier_ranked[:5]:
    n_in_list = len(concept_neighbors.get(c, set()) & candidate_set)
    is_gold = "GOLD" if c in gold_ids else ""
    print(f"  {concept_pool_lookup[c]['term']}: {n_in_list} neighbors in list {is_gold}")

Document: val_v00378
Gold: ['HIGH RISE FLATS']

'HIGH RISE FLATS': Qwen3 rank 1 -> Hierarchy-aware rank 6

Qwen3 top 5: ['HIGH RISE FLATS', 'HOUSING FOR THE ELDERLY', 'RESIDENTIAL CARE OF THE ELDERLY', 'URBAN POPULATION', 'HOUSING POLICY']
Hierarchy-aware top 5: ['HOUSING', 'URBAN POPULATION', 'RESIDENTIAL CARE OF THE ELDERLY', 'HOUSING FOR THE ELDERLY', 'HOUSING POLICY']

Neighbor-in-list counts for hierarchy-aware top 5:
  HOUSING: 14 neighbors in list 
  URBAN POPULATION: 3 neighbors in list 
  RESIDENTIAL CARE OF THE ELDERLY: 2 neighbors in list 
  HOUSING FOR THE ELDERLY: 1 neighbors in list 
  HOUSING POLICY: 3 neighbors in list 


### 3.6 Variant 2 - Normalizing by connectivity
> The traced example suggests a hub-bias problem: broad concepts with many neighbors get boosted regardless of relevance. Testing the direct fix normalizing the boost by each concept's total connectivity, so being one of many neighbors in a large hub doesn't count for as much as being one of few for a specific concept.

In [13]:
def hierarchy_rerank_normalized(candidates, neighbor_boost_weight=0.3):
    n = len(candidates)
    candidate_set = set(candidates)
    base_scores = {cid: (n - i) for i, cid in enumerate(candidates)}

    boosted_scores = {}
    for cid in candidates:
        total_neighbors = concept_neighbors.get(cid, set())
        neighbors_in_list = total_neighbors & candidate_set

        fraction = len(neighbors_in_list) / len(total_neighbors) if total_neighbors else 0
        boost = fraction * neighbor_boost_weight * n
        boosted_scores[cid] = base_scores[cid] + boost

    return sorted(candidates, key=lambda cid: boosted_scores[cid], reverse=True)


def compute_hierarchy_aware_normalized():
    predictions = {}
    for doc_id in val_ids:
        candidates = qwen_predictions[doc_id]
        predictions[doc_id] = hierarchy_rerank_normalized(candidates)
    metrics = evaluate_retrieval(predictions, gold)
    with open(os.path.join(RESULTS_DIR, 'hierarchy_aware_normalized_predictions.json'), 'w') as f:
        json.dump(predictions, f)
    return metrics

hierarchy_norm_metrics = run_or_load('hierarchy_aware_normalized_retrieval', compute_hierarchy_aware_normalized)
print(hierarchy_norm_metrics)

Loaded cached result: hierarchy_aware_normalized_retrieval
{'MRR': 0.3116342752059202, 'Recall@5': 0.24673721340388008, 'Recall@10': 0.35617283950617284, 'NDCG@10': 0.2517635956985084}


### 3.7 Variant 3 - Restricting to broader/narrower relations
> Normalizing didn't fix the drop, ruling out hub-bias as the sole cause. Testing whether the noisier 'related' relation the vaguest of ELSST's three relation type is contributing noise, by boosting only on the stricter broader/narrower relations.

In [14]:
concept_neighbors_strict = {}
for cid, c in concept_pool_lookup.items():
    term = c['term'].upper().strip()
    node_id = term_to_node.get(term)
    if node_id is None:
        concept_neighbors_strict[cid] = set()
        continue
    node = concept_nodes[node_id]
    neighbor_node_ids = node['broader'] + node['narrower']  # dropped 'related'
    neighbor_concept_ids = set()
    for nid in neighbor_node_ids:
        if nid in concept_nodes:
            neighbor_term = concept_nodes[nid]['term']
            if neighbor_term in term_to_concept_id:
                neighbor_concept_ids.add(term_to_concept_id[neighbor_term])
    concept_neighbors_strict[cid] = neighbor_concept_ids

def hierarchy_rerank_strict(candidates, neighbor_boost_weight=0.3):
    n = len(candidates)
    candidate_set = set(candidates)
    base_scores = {cid: (n - i) for i, cid in enumerate(candidates)}
    boosted_scores = {}
    for cid in candidates:
        neighbors_in_list = concept_neighbors_strict.get(cid, set()) & candidate_set
        boost = len(neighbors_in_list) * neighbor_boost_weight * (n / 10)
        boosted_scores[cid] = base_scores[cid] + boost
    return sorted(candidates, key=lambda cid: boosted_scores[cid], reverse=True)

def compute_hierarchy_strict():
    predictions = {doc_id: hierarchy_rerank_strict(qwen_predictions[doc_id]) for doc_id in val_ids}
    metrics = evaluate_retrieval(predictions, gold)
    with open(os.path.join(RESULTS_DIR, 'hierarchy_strict_predictions.json'), 'w') as f:
        json.dump(predictions, f)
    return metrics

print(run_or_load('hierarchy_strict_retrieval', compute_hierarchy_strict))

Loaded cached result: hierarchy_strict_retrieval
{'MRR': 0.342110037399184, 'Recall@5': 0.28328924162257496, 'Recall@10': 0.3630731922398589, 'NDCG@10': 0.2713082423989227}


### 3.8 Variant 4 - Specificity penalty
> Restricting relation types recovered some ground but not enough. Reframing the whole approach: instead of boosting candidates for co-occurring with others in the list, directly penalize concepts with many total connections a specificity prior derived from the same hierarchy data, targeting the traced failure mode directly rather than indirectly.

In [15]:
def hierarchy_specificity_rerank(candidates, penalty_weight=0.15):
    n = len(candidates)
    base_scores = {cid: (n - i) for i, cid in enumerate(candidates)}

    scores = {}
    for cid in candidates:

        total_connections = len(concept_neighbors_strict.get(cid, set()))
        penalty = penalty_weight * total_connections
        scores[cid] = base_scores[cid] - penalty

    return sorted(candidates, key=lambda cid: scores[cid], reverse=True)

def compute_hierarchy_specificity():
    predictions = {doc_id: hierarchy_specificity_rerank(qwen_predictions[doc_id]) for doc_id in val_ids}
    metrics = evaluate_retrieval(predictions, gold)
    with open(os.path.join(RESULTS_DIR, 'hierarchy_specificity_predictions.json'), 'w') as f:
        json.dump(predictions, f)
    return metrics

print(run_or_load('hierarchy_specificity_retrieval', compute_hierarchy_specificity))

Loaded cached result: hierarchy_specificity_retrieval
{'MRR': 0.3668535174730653, 'Recall@5': 0.2752204585537919, 'Recall@10': 0.36073633156966495, 'NDCG@10': 0.28164553640921053}


### 3.9 Key Findings

1. **Naive co-occurrence boosting** raw neighbor count within the candidate list
significantly underperformed Qwen3 alone MRR 0.321 vs 0.368, p<0.001 traced to a
specific bias favoring broad "hub" concepts e.g. HOUSING, with 14 in-list neighbors
over the correct, more specific answer HIGH RISE FLATS.

2. **Normalizing the boost by total connectivity** did not fix this MRR 0.312 ruling out hub bias as the sole cause.

3. **Restricting to broader, narrower relations only** dropping the noisier "related" relation recovered some performance MRR 0.342 but remained significantly below baseline.

4. **Reframing hierarchy structure as a specificity penalty** rather than a
co-occurrence boost reached genuine statistical parity with Qwen3 MRR 0.367 vs
0.368, Wilcoxon p=0.134, despite changing 572/756 rankings.

**Conclusion:** hierarchy structure alone does not surpass strong embedding retrieval on this task when naively applied, but a correctly designed specificity prior matches it suggesting the value of hierarchy information here lies in filtering overly generic
candidates, not in reinforcing thematic co-occurrence, which the embedding model's own
top-50 selection already captures.

In [16]:
hierarchy_export_path = os.path.join(RESULTS_DIR, 'concept_hierarchy_strict.json')
with open(hierarchy_export_path, 'w') as f:
    json.dump({cid: list(neighbors) for cid, neighbors in concept_neighbors_strict.items()}, f)
print(f"Saved hierarchy data for {len(concept_neighbors_strict)} concepts")

Saved hierarchy data for 3433 concepts
